# Tutorial: MAGMA Gene-Set and Gene Property Analysis

This tutorial demonstrates the MAGMA functions added to `cellink` for cell-type enrichment
analysis using GWAS summary statistics:

1. **MAGMA gene-set analysis (GSA)** — test whether top-scoring genes per cell type are enriched for GWAS signal.
2. **MAGMA gene property analysis (GPA)** — test the linear relationship between continuous per-gene scores and GWAS gene-level z-scores.

All functions are available at `cellink.tl.external`. MAGMA must be installed separately; download it from [https://ctg.cncr.nl/software/magma](https://ctg.cncr.nl/software/magma).

### What is a cell-type specificity score?

The central input to both analyses is a **specificity score**: for each gene and each cell type, a number capturing how selectively that gene is expressed in that cell type relative to all others. A gene with high specificity for, say, CD8 Naive T cells is expressed much more in that cell type than anywhere else; a gene with low specificity is expressed broadly across many cell types and isn't very informative about CD8 Naive biology specifically.

This tutorial computes specificity using the method from [Duncan et al. 2025](https://www.nature.com/articles/s41593-024-01834-w): for each gene, take its total expression across all cell types and compute what *fraction* of that total comes from each individual cell type. A gene's specificity scores across cell types therefore sum to 1 — a gene expressed equally everywhere has low specificity everywhere, while a gene expressed almost exclusively in one cell type has a specificity score near 1 there and near 0 elsewhere. The working assumption is that genes highly specific to a cell type are more likely to be the genes through which that cell type's biology contributes to disease risk — so testing whether GWAS signal concentrates near these genes is how we link a cell type to a trait.

> **Where this connects to LDSC:** these same specificity scores were originally developed for stratified LD-score regression (S-LDSC), a separate method for partitioning trait heritability across annotations using GWAS summary statistics and a population reference panel. `cellink`'s `preprocess_for_sldsc` function computes the scores (hence the name), but the scores themselves — and everything in this notebook — are used independently of LDSC; MAGMA's gene-set and gene-property tests consume them directly. If you want the full S-LDSC heritability-partitioning workflow instead, see the companion notebook `cell_level_ldsc_analysis_updates.ipynb`.

### GSA vs GPA — which one should I use?

Both tests start from the same specificity scores but ask the question differently:

- **GSA** thresholds each cell type's scores to a top-N% gene set (binary: in the set or not) and tests whether that set is enriched for GWAS signal. Simple to interpret, mirrors a standard binary enrichment test, and a good default — especially if you want results comparable to a published top-10%-style analysis.
- **GPA** uses the full continuous score for every gene (no thresholding) and tests the linear relationship between score and GWAS gene-level association. It avoids the arbitrary top-N% cutoff and can be more powerful when specificity varies smoothly across genes rather than splitting cleanly into "specific" vs "not."

If you're unsure, start with GSA for its simplicity; switch to GPA if a hard top-N% cutoff feels like it's throwing away real signal in your data.

### Required inputs

| Input | Description |
|---|---|
| `specificity_df` | Specific genes × cell-types DataFrame of per-gene scores (e.g. from `preprocess_for_sldsc`) |
| Gene coordinate file | Tab-separated file mapping genes to chr/start/end positions |
| GWAS SNP location file (for MAGMA Step I) | SNP/CHR/BP columns |
| MAGMA gene location file | Fetched from Ensembl via `get_magma_gene_loc` |
| PLINK bfile (for MAGMA Step II) | LD reference panel, e.g. 1000 Genomes EUR |
| GWAS p-value file (for MAGMA Step II) | SNP + P columns |
| `.genes.raw` file (for MAGMA Step III) | Output of Step II; can be pre-computed |


## Environment Setup

In [ ]:
import requests, zipfile, io, os, stat

# Download MAGMA v1.10 (Linux, 64-bit, static linking)
# Official source / full download table: https://cncr.nl/research/magma/
url = "https://vu.data.surf.nl/index.php/s/lxDgt2dNdNr6DYt/download"

r = requests.get(url)
r.raise_for_status()

with zipfile.ZipFile(io.BytesIO(r.content)) as z:
    z.extractall("magma_bin")

# Make all extracted files executable
for root, dirs, files in os.walk("magma_bin"):
    for file in files:
        fpath = os.path.join(root, file)
        st = os.stat(fpath)
        os.chmod(fpath, st.st_mode | stat.S_IEXEC)


In [ ]:
import cellink
import numpy as np
import pandas as pd
from pathlib import Path
import os

from cellink._core import DAnn, GAnn
from cellink.resources import get_onek1k, get_gwas_catalog_study_summary_stats
from cellink.tl.external import (
    preprocess_for_sldsc,
    generate_sldsc_genesets,
    scores_to_gmt,
    scores_to_covar,
    genesets_dir_to_entrez_gmt,
    run_magma_annotate,
    get_magma_gene_loc,
    run_magma_gene_analysis,
    run_magma_gsa,
    run_magma_gpa,
)
from cellink.resources import get_1000genomes_plink_files

# Path to the downloaded MAGMA binary (extracted by the cell above)
MAGMA_BIN = os.path.join("magma_bin", "magma")

# MAGMA gene location file — fetched from Ensembl and cached in ~/cellink_data/magma/
MAGMA_GENE_LOC = get_magma_gene_loc(genome_build="GRCh37")

# 1000G PLINK files — downloaded and cached in ~/cellink_data/1000genomes_plink_EUR/
_plink_path, _plink_prefix = get_1000genomes_plink_files(
    config_path=str(Path(cellink.__file__).parent / "resources" / "config" / "1000genomes.yaml"),
    population="EUR",
)
# Use chr22 as the LD reference for the tutorial (smallest autosome; use all chroms in production)
G1000_BFILE = str(_plink_path / f"{_plink_prefix}22")

# Cell type for single-cell examples
CELL_TYPE = "CD8 Naive"
celltype_key = "predicted.celltype.l2"
original_donor_col = "donor_id"


## Load and Prepare Data

We load the real OneK1K dataset and compute cell-type specificity scores as described above. (If you've already run the companion LDSC notebook, these preprocessing steps are identical — only the downstream analysis differs.)


In [ ]:
dd = get_onek1k()
print(f"Dataset shape: {dd.shape}")
dd.C.obs[DAnn.donor] = dd.C.obs[original_donor_col]


In [ ]:
# Real OneK1K has ~1.25M cells across ~36k genes — far more than this
# tutorial needs for per-cell-type specificity estimation, and too large to
# copy/preprocess in memory as-is. Subsample cells for speed (real analyses
# would use the full dataset).
np.random.seed(0)
n_cells_keep = 30_000
cell_idx = np.sort(np.random.choice(dd.shape[2], min(n_cells_keep, dd.shape[2]), replace=False))
dd = dd[:, :, cell_idx, :].copy()
print(f"Subsampled to {dd.shape[2]:,} cells")


In [ ]:
dd.C.var["gene"] = dd.C.var_names
adata = dd.C.copy()

adata_filtered, mean_expr, specificity = preprocess_for_sldsc(
    adata,
    celltype_col=celltype_key,
    gene_identifier_mode="ensembl",
    gene_col="gene",
    genome_build="GRCh37",
    inplace=False,
)
print(f"Specificity shape: {specificity.shape}")
specificity.head()

---
## Part 1 — MAGMA 

MAGMA provides a complementary enrichment analysis that does not require LD score files. The pipeline has three steps:

| Step | Function | Output |
|---|---|---|
| I — Annotate | `run_magma_annotate` | `.genes.annot` |
| II — Gene analysis | `run_magma_gene_analysis` | `.genes.raw` |
| III-a — Gene-set analysis | `run_magma_gsa` | `.gsa.out` |
| III-b — Gene property analysis | `run_magma_gpa` | `.gsa.out` |

Steps I and II only need to be run once per GWAS. Step III is run once per score set / method.

### Step I — Annotate SNPs to genes

In [ ]:
# Derive MAGMA's SNP location file (SNP / CHR / BP) from real GWAS summary
# statistics — the same study (GCST004787, coronary artery disease) used in
# the companion LDSC notebook's heritability/genetic-correlation sections.
gwas_summary_statistic_path = get_gwas_catalog_study_summary_stats("GCST004787", genome_build="GRCh37", return_path=True)
gwas_df = pd.read_csv(gwas_summary_statistic_path, sep="\t")
gwas_df = gwas_df.dropna(subset=["variant_id", "chromosome", "base_pair_location", "p_value"])
# chromosome/base_pair_location load as float64 (NaN forces the column to
# float dtype) — cast to int before filtering/writing, otherwise "22.0" never
# matches "22" and MAGMA receives malformed CHR/BP columns.
gwas_df["chromosome"] = gwas_df["chromosome"].astype(int)
gwas_df["base_pair_location"] = gwas_df["base_pair_location"].astype(int)
gwas_df = gwas_df[gwas_df["chromosome"] == 22]   # restrict to chr22, matching G1000_BFILE
print(f"{len(gwas_df):,} chr22 SNPs with usable GWAS data")

os.makedirs("magma_results", exist_ok=True)
GWAS_SNP_LOC = "magma_results/gwas_snp_loc.txt"
gwas_df[["variant_id", "chromosome", "base_pair_location"]].rename(
    columns={"variant_id": "SNP", "chromosome": "CHR", "base_pair_location": "BP"}
).to_csv(GWAS_SNP_LOC, sep="\t", index=False)

annot_result = run_magma_annotate(
    snp_loc=GWAS_SNP_LOC,
    gene_loc=MAGMA_GENE_LOC,
    out_prefix="magma_results/cad",
    magma_bin=MAGMA_BIN,
    window_kb=0,   # gene body only; use e.g. 35 for ±35 kb
)
print("Annotation file:", annot_result["annot_file"])


In [ ]:
# Preview the command without running:
cmd_preview = run_magma_annotate(
    snp_loc=GWAS_SNP_LOC,
    gene_loc=MAGMA_GENE_LOC,
    out_prefix="magma_results/cad",
    magma_bin=MAGMA_BIN,
    run=False,
)
print(" ".join(cmd_preview["command"]))


### Step II — Gene-level association analysis

This step uses a PLINK LD reference panel to compute gene-level z-scores from the GWAS p-values. The output `.genes.raw` file is the input for both GSA and GPA.

In [ ]:
# GWAS p-value file: tab-delimited, columns SNP / P / N. The real summary
# statistics already carry a per-SNP sample size (n_samples), so we pass
# that through directly instead of a single global N.
GWAS_PVAL_FILE = "magma_results/gwas_pval.txt"
gwas_pval_df = gwas_df[["variant_id", "p_value", "n_samples"]].copy()
gwas_pval_df["n_samples"] = gwas_pval_df["n_samples"].astype(int)
gwas_pval_df.rename(
    columns={"variant_id": "SNP", "p_value": "P", "n_samples": "N"}
).to_csv(GWAS_PVAL_FILE, sep="\t", index=False)

gene_result = run_magma_gene_analysis(
    bfile=G1000_BFILE,
    pval_file=GWAS_PVAL_FILE,
    gene_annot=annot_result["annot_file"],
    out_prefix="magma_results/cad",
    ncol="N",   # tells MAGMA to read per-SNP N from the file's "N" column
    magma_bin=MAGMA_BIN,
)
print("Gene results:", gene_result["gene_results"])


In [ ]:
# If you already have a pre-computed .genes.raw (e.g. from the combined_pipeline)
# you can skip Steps I and II and point directly to it. We just computed one
# for real above, so we use that:
GENES_RAW = gene_result["gene_results"]


---
## Part 2 — MAGMA Gene-Set Analysis (GSA)

GSA tests whether the top-scoring genes in each cell type show stronger GWAS association than background. There are two paths to create the required GMT file:

* **From a scores DataFrame** — use `scores_to_gmt` (selects top-N% per cell type)
* **From existing LDSC `.GeneSet` files** — use `genesets_dir_to_entrez_gmt`

### Path A — From continuous scores

In [ ]:
# specificity is a genes × cell-types DataFrame with ENSG IDs as the index
gmt_file = scores_to_gmt(
    scores=specificity,
    out_file="magma_gsa/onek1k_specificity_top10.gmt",
    top_frac=0.10,          # top 10% genes per cell type
    ascending=False,         # ascending=True would select the bottom 10%
    set_name_prefix="onek1k_specificity",  # optional prefix for set names
)
print("GMT file:", gmt_file)

# Preview the first two lines
with open(gmt_file) as f:
    for i, line in enumerate(f):
        parts = line.rstrip().split("\t")
        print(f"Set '{parts[0]}': {len(parts)-2} genes")
        if i == 1:
            break

In [ ]:
# If gene scores use gene symbols rather than ENSG IDs, provide a mapping:
# gene_map can be a path to a TSV with columns gene_name / ensg_id,
# or a pd.Series indexed by symbol with ENSG values.

# gmt_file = scores_to_gmt(
#     scores=scores_with_symbols,
#     out_file="magma_gsa/scores_mapped.gmt",
#     gene_map="cellink/combined_pipeline/gene_name_to_ensg.tsv",
# )

### Path B — From existing binary LDSC `.GeneSet` files

In [ ]:
# First generate the *.GeneSet files (one per cell type, top 10% specificity using method from )
from cellink.tl.external import generate_sldsc_genesets
generate_sldsc_genesets(specificity, dd.C, out_dir="ldsc_genesets", top_frac=0.10, overwrite=True)

# Converts the *.GeneSet files produced by generate_sldsc_genesets into GMT.
# Gene IDs are passed through as-is (no Ensembl → Entrez conversion).
gmt_from_genesets = genesets_dir_to_entrez_gmt(
    geneset_dir="ldsc_genesets",
    out_gmt="magma_gsa/ldsc_genesets.gmt",
    include_control=False,
)
print("GMT from GeneSet files:", gmt_from_genesets)

### Run GSA

In [ ]:
gsa_result = run_magma_gsa(
    gene_results=GENES_RAW,
    set_annot=str(gmt_file),
    out_prefix="magma_gsa/cad_onek1k_specificity",
    magma_bin=MAGMA_BIN,
)
print("GSA results:", gsa_result["results_file"])

# Load and display results
gsa_df = pd.read_csv(gsa_result["results_file"], sep=r"\s+", comment="#")
gsa_df.sort_values("P").head(10)

In [ ]:
# Extra MAGMA flags are forwarded via **kwargs, e.g. joint competitive test:
# run_magma_gsa(..., model="multi")

---
## Part 3 — MAGMA Gene Property Analysis (GPA)

GPA tests the linear relationship between continuous per-gene scores and GWAS gene z-scores. Unlike GSA it uses all genes with scores — no top-N threshold. The covariate file contains one column per cell type.

### Build the covariate file

In [ ]:
covar_file = scores_to_covar(
    scores=specificity,
    out_file="magma_gpa/onek1k_specificity.covar",
    negate=False,   # set negate=True to test enrichment in low-score genes
)
print("Covariate file:", covar_file)

# Preview
pd.read_csv(covar_file, sep="\t", index_col=0).head()

In [ ]:
# Gene symbols → ENSG mapping works the same way as in scores_to_gmt:
# covar_file = scores_to_covar(
#     scores=scores_with_symbols,
#     out_file="magma_gpa/scores_mapped.covar",
#     gene_map="cellink/combined_pipeline/gene_name_to_ensg.tsv",
# )

### Joint GPA (default)

All cell types are tested in a single MAGMA call. MAGMA may drop highly correlated covariates via its collinearity filter. Use univariate mode if this is a concern.

In [ ]:
gpa_result = run_magma_gpa(
    gene_results=GENES_RAW,
    gene_covar=str(covar_file),
    out_prefix="magma_gpa/cad_onek1k_specificity_joint",
    magma_bin=MAGMA_BIN,
    univariate=False,
)
print("GPA results:", gpa_result["results_file"])

gpa_df = pd.read_csv(gpa_result["results_file"], sep=r"\s+", comment="#")
gpa_df.sort_values("P").head(10)

### Univariate GPA

Each cell type is tested in an independent MAGMA call. Slower than joint mode but guarantees a result for every cell type — recommended when covariate columns are highly correlated (e.g. residual CV scores or negated scores).

In [ ]:
gpa_univ_result = run_magma_gpa(
    gene_results=GENES_RAW,
    gene_covar=str(covar_file),
    out_prefix="magma_gpa/cad_onek1k_specificity_univariate",
    magma_bin=MAGMA_BIN,
    univariate=True,
)
print("Univariate GPA results:", gpa_univ_result["results_file"])

gpa_univ_df = pd.read_csv(gpa_univ_result["results_file"], sep=r"\s+", comment="#")
gpa_univ_df.sort_values("P").head(10)

---
## Part 4 — LDSC → MAGMA: Full Workflows

The two complete workflows from LDSC score outputs to MAGMA results.

### Workflow A: LDSC binary genesets → MAGMA GSA

In [ ]:
# 1. Generate LDSC gene sets (top 10% specificity per cell type)
generate_sldsc_genesets(specificity, dd.C, out_dir="ldsc_genesets", top_frac=0.10, overwrite=True)

# 2. Convert .GeneSet → GMT (MAGMA format, ENSG IDs kept as-is)
gmt_ldsc = genesets_dir_to_entrez_gmt(
    geneset_dir="ldsc_genesets",
    out_gmt="magma_gsa/ldsc_genesets_workflow_a.gmt",
)

# 3. GSA
gsa_a = run_magma_gsa(
    gene_results=GENES_RAW,
    set_annot=str(gmt_ldsc),
    out_prefix="magma_gsa/workflow_a_cad",
    magma_bin=MAGMA_BIN,
)

results_a = pd.read_csv(gsa_a["results_file"], sep=r"\s+", comment="#")
print(results_a.sort_values("P").head())

### Workflow B: Continuous scores → MAGMA GPA

In [ ]:
# 1. Build covariate file directly from the specificity score matrix
covar_b = scores_to_covar(
    scores=specificity,
    out_file="magma_gpa/onek1k_specificity_workflow_b.covar",
)

# 2. GPA (joint mode)
gpa_b = run_magma_gpa(
    gene_results=GENES_RAW,
    gene_covar=str(covar_b),
    out_prefix="magma_gpa/workflow_b_cad",
    magma_bin=MAGMA_BIN,
)

results_b = pd.read_csv(gpa_b["results_file"], sep=r"\s+", comment="#")
print(results_b.sort_values("P").head())

---
## Part 5 — Tips and Common Patterns

### Previewing commands without running

All MAGMA functions accept `run=False` to return the command list without executing it. Useful for submitting to a cluster.

In [ ]:
for fn_name, result in [
    ("run_magma_annotate",
     run_magma_annotate(snp_loc=GWAS_SNP_LOC, gene_loc=MAGMA_GENE_LOC,
                        out_prefix="magma_results/cad", magma_bin=MAGMA_BIN, run=False)),
    ("run_magma_gene_analysis",
     run_magma_gene_analysis(bfile=G1000_BFILE, pval_file=GWAS_PVAL_FILE,
                             gene_annot="magma_results/cad.genes.annot",
                             out_prefix="magma_results/cad",
                             ncol="N", magma_bin=MAGMA_BIN, run=False)),
    ("run_magma_gsa",
     run_magma_gsa(gene_results=GENES_RAW, set_annot=str(gmt_file),
                   out_prefix="magma_gsa/preview", magma_bin=MAGMA_BIN, run=False)),
    ("run_magma_gpa",
     run_magma_gpa(gene_results=GENES_RAW, gene_covar=str(covar_file),
                   out_prefix="magma_gpa/preview", magma_bin=MAGMA_BIN, run=False)),
]:
    print(f"{fn_name}:\n  {' '.join(result['command'])}\n")

### Negating scores to test low-score enrichment

Some metrics (e.g. coefficient of variation) are expected to be *low* in disease-relevant genes. Pass `negate=True` to `scores_to_covar`, or `ascending=True` to `scores_to_gmt`, to select/test genes with the *lowest* scores:

In [ ]:
# GSA: bottom 10% genes (lowest CV)
gmt_bottom = scores_to_gmt(
    scores=specificity,
    out_file="magma_gsa/specificity_bottom10.gmt",
    top_frac=0.10,
    ascending=True,   # select lowest-scoring genes
)

# GPA: negate scores so low-score genes get high covariate values
covar_neg = scores_to_covar(
    scores=specificity,
    out_file="magma_gpa/specificity_negated.covar",
    negate=True,
)

### Passing extra MAGMA flags

Any MAGMA option not explicitly supported can be forwarded via `**kwargs`. Each `key=value` pair becomes `--key value` in the MAGMA call:

In [ ]:
# Competitive gene-set test (MAGMA's "multi" model)
run_magma_gsa(
    gene_results=GENES_RAW,
    set_annot=str(gmt_file),
    out_prefix="magma_gsa/cad_competitive",
    magma_bin=MAGMA_BIN,
    model="multi",   # forwarded as --model multi
    run=False,       # preview only
)

---
## Summary

This tutorial demonstrated all new functions added to `cellink.tl.external`:

| Function | Purpose |
|---|---|
| `scores_to_gmt` | Convert score DataFrame → MAGMA GMT (top-N% per cell type) |
| `scores_to_covar` | Convert score DataFrame → MAGMA `.covar` (all genes, continuous) |
| `genesets_dir_to_entrez_gmt` | Convert LDSC `.GeneSet` files → MAGMA GMT |
| `run_magma_annotate` | MAGMA Step I: SNP → gene annotation |
| `run_magma_gene_analysis` | MAGMA Step II: gene-level association analysis |
| `run_magma_gsa` | MAGMA Step III-a: gene-set analysis (binary) |
| `run_magma_gpa` | MAGMA Step III-b: gene property analysis (continuous), with optional univariate mode |

Key properties shared across all functions:

* **`run=False`** returns the command list without executing — useful for HPC job submission.
* **Gene ID handling** — `scores_to_gmt` and `scores_to_covar` accept a `gene_map` argument for symbol → ENSG translation; rows that cannot be mapped to ENSG are dropped.
* **Univariate GPA** — `run_magma_gpa(univariate=True)` tests each cell type independently, avoiding MAGMA's collinearity filter when scores are highly correlated.